# CS-781 Phase 2 — Experiment 1 (v3 BEST)
## Model 4 + **Full Fine-Tuning** for Legal NER
### BILOU token head + bidirectional task adaptation + constrained Viterbi + per-label Macro-F1 calibration

**Start:** your own Phase-1 Model 4 (`efficient_gqa_rope_swiglu_decoder`, ~99.31M params).  
**Experiment identity remains unchanged:** **Model 4 + full fine-tuning** of the complete Phase-1 backbone plus an NER classification head.

### Improvements retained from v2
1. **Bidirectional attention for NER** (`BIDIRECTIONAL_ATTENTION=True`). The Phase-1 weights are unchanged at initialization; for NER, tokens can use both left and right context.
2. **Constrained BILOU Viterbi decoding** to prevent illegal tag sequences.
3. **Central-window logit selection** for overlapping long-document windows.
4. **Tokenizer/checkpoint LM sanity test** before expensive training.
5. **Oracle span-ceiling test** before training.
6. Faster entity/window-boundary handling.

### Additional v3 improvements
7. **Exact tokenizer SHA-256 check** against the tokenizer you verified for Model 4.
8. **Kaggle `HF_TOKEN` secret support** with public-download fallback.
9. **Per-label confidence calibration** on the held-out fold. The official metric is Macro-F1, so each entity type gets its own held-out confidence threshold rather than forcing one global threshold on all 12 classes.
10. Global-threshold score is still reported alongside calibrated score, so the gain is transparent rather than hidden.
11. More metadata is stored in checkpoints for later phases/viva reproducibility.

### Competition-safe design
- No external pretrained LM is used.
- Only your own Phase-1 Model 4 initializes the backbone.
- Only official Phase-2 data is used for NER supervision.
- Fold 1 is training and Fold 2 is held out for model/threshold selection.
- Exact character spans are reconstructed and scored with the official strict metric.
- Optional final refit uses both public annotated folds only after epoch/threshold selection.

> Keep `DO_FINAL_REFIT_ON_ALL=False` for the first held-out run. Do not refit until the held-out result looks healthy.


In [1]:
import os, sys, json, math, time, random, re, zipfile, subprocess, gc, hashlib
from pathlib import Path
from collections import Counter, defaultdict

try:
    import huggingface_hub
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download

SEED=781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
print("PyTorch:",torch.__version__)
print("Device:",DEVICE)
if DEVICE=="cuda": print("GPU:",torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [37]:
ROOT=Path("/kaggle/working/phase2_exp1_model4_fullft")
ROOT.mkdir(parents=True,exist_ok=True)
HF_REPO="Exploration-Lab/CS-781-Capstone"

MODEL4_BEST_PATH=None
TOKENIZER_PATH=None

NER_SEQ_LEN=768
WINDOW_OVERLAP=192
WINDOW_STEP=NER_SEQ_LEN-WINDOW_OVERLAP

EPOCHS=6
TRAIN_BATCH_SIZE=2
EVAL_BATCH_SIZE=4
GRAD_ACCUM_STEPS=8
BACKBONE_LR=5e-5
HEAD_LR=5e-4
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.10
MAX_GRAD_NORM=1.0
HEAD_DROPOUT=0.10
USE_CLASS_WEIGHTS=True
AMP=True

THRESHOLD_GRID=[round(x,2) for x in np.arange(0.00,0.81,0.05)]
DO_FINAL_REFIT_ON_ALL=True

# ---- v2 switches ----
BIDIRECTIONAL_ATTENTION=True   # True: drop the causal mask for NER (recommended). False: causal ablation.
USE_VITERBI=True               # constrained BILOU decoding
PRESPLIT_PUNCT=True          # True: split punctuation from words before BPE (changes token units vs Phase-1!)
TOKENIZER_CHECK_WINDOWS=24     # windows used for the LM-loss tokenizer sanity check
USE_PER_LABEL_THRESHOLDS=True  # Recommended for the official Macro-F1 objective.
# SHA-256 of the tokenizer.json you verified after regenerating the Phase-1 tokenizer.
EXPECTED_TOKENIZER_SHA256="9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

BEST_NER_PATH=ROOT/"best_ner.pt"
VAL_PRED_PATH=ROOT/"val_prediction.json"
TEST_PRED_PATH=ROOT/"prediction.json"
SUBMISSION_ZIP=ROOT/"submission.zip"

print("ROOT:",ROOT)
print("NER seq len:",NER_SEQ_LEN,"overlap:",WINDOW_OVERLAP,"step:",WINDOW_STEP)


ROOT: /kaggle/working/phase2_exp1_model4_fullft
NER seq len: 768 overlap: 192 step: 576


In [20]:
def find_files():
    ckpts,toks=[],[]
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.exists(base): continue
        for root,dirs,files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn); low=fn.lower()
                if fn.endswith(".pt") and "eff_full_best" in low: ckpts.append(p)
                if low=="tokenizer.json" or ("tokenizer" in low and low.endswith(".json")): toks.append(p)
    return sorted(ckpts),sorted(toks)

ckpts,toks=find_files()
print("Checkpoint candidates:")
for x in ckpts: print(" ",x)
print("\nTokenizer candidates:")
for x in toks: print(" ",x)

if MODEL4_BEST_PATH is None:
    preferred=[x for x in ckpts if "phase1-model4-decoder-efficient" in x]
    if preferred: MODEL4_BEST_PATH=preferred[-1]
    elif ckpts: MODEL4_BEST_PATH=ckpts[-1]

if TOKENIZER_PATH is None and len(toks)==1:
    TOKENIZER_PATH=toks[0]
elif TOKENIZER_PATH is None and len(toks)>1:
    print("\n⚠️ Multiple tokenizers found. Set TOKENIZER_PATH manually in Cell 2.")

assert MODEL4_BEST_PATH and os.path.exists(MODEL4_BEST_PATH), "Set MODEL4_BEST_PATH manually."
assert TOKENIZER_PATH and os.path.exists(TOKENIZER_PATH), (
    "Exact Phase-1 tokenizer.json not found. Regenerate/save the original tokenizer from the Phase-1 notebook, "
    "attach it here, and set TOKENIZER_PATH."
)
print("\nUsing checkpoint:",MODEL4_BEST_PATH)
print("Using tokenizer:",TOKENIZER_PATH)


def sha256_file(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

tok_sha=sha256_file(TOKENIZER_PATH)
print("Tokenizer SHA-256:",tok_sha)
if EXPECTED_TOKENIZER_SHA256:
    assert tok_sha==EXPECTED_TOKENIZER_SHA256, (
        "❌ tokenizer.json differs from the verified Phase-1 tokenizer. "
        f"Expected {EXPECTED_TOKENIZER_SHA256}, got {tok_sha}"
    )
    print("✅ tokenizer.json is byte-for-byte identical to the verified copy.")


Checkpoint candidates:
  /kaggle/input/datasets/aaryant23/phase2-model-wts-tokenizer/eff_full_best (6).pt

Tokenizer candidates:
  /kaggle/input/datasets/aaryant23/phase2-model-wts-tokenizer/tokenizer.json

Using checkpoint: /kaggle/input/datasets/aaryant23/phase2-model-wts-tokenizer/eff_full_best (6).pt
Using tokenizer: /kaggle/input/datasets/aaryant23/phase2-model-wts-tokenizer/tokenizer.json
Tokenizer SHA-256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d
✅ tokenizer.json is byte-for-byte identical to the verified copy.


In [22]:
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]

class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]; self.itos=list(BPE_SPECIALS)
        self.stoi={t:i for i,t in enumerate(self.itos)}; self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word); ranks=self._ranks
        while len(symbols)>1:
            best=None; best_rank=None
            for j in range(len(symbols)-1):
                r=ranks.get((symbols[j],symbols[j+1]))
                if r is not None and (best_rank is None or r<best_rank):
                    best=(symbols[j],symbols[j+1]); best_rank=r
            if best is None: break
            left,right=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==left and symbols[j+1]==right:
                    merged.append(left+right); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",text):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok in BPE_SPECIALS: continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        obj=json.load(open(path,encoding="utf-8")); tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]; tok.itos=list(obj["itos"])
        tok.stoi={t:i for i,t in enumerate(tok.itos)}; tok.vocab_size=len(tok.itos)
        tok.symbol_bytes={}
        for b in range(256): tok.symbol_bytes[f"{b:02x}"]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_PATH)
print("Tokenizer vocab:",len(tokenizer))

UNIT_RE=re.compile(r"\w+|[^\w\s]|\s+" if PRESPLIT_PUNCT else r"\S+|\s+")
print("Token unit regex:",UNIT_RE.pattern)

def encode_with_char_offsets(text):
    ids=[]; offsets=[]; exact=[]; unk=tokenizer.unk_token_id
    for m in UNIT_RE.finditer(text):
        unit=m.group(0); base=m.start(); symbols=tokenizer.encode_word(unit)
        byte_to_char={0:0}; bpos=0
        for ci,ch in enumerate(unit):
            bpos+=len(ch.encode("utf-8")); byte_to_char[bpos]=ci+1
        boundaries=sorted(byte_to_char)
        sb=0
        for sym in symbols:
            raw=tokenizer.symbol_bytes.get(sym)
            if raw is None:
                ids.append(unk); offsets.append((m.start(),m.end())); exact.append(False); continue
            eb=sb+len(raw); ok=(sb in byte_to_char and eb in byte_to_char)
            if ok:
                cs=base+byte_to_char[sb]; ce=base+byte_to_char[eb]
            else:
                lb=max(x for x in boundaries if x<=sb); rb=min(x for x in boundaries if x>=eb)
                cs=base+byte_to_char[lb]; ce=base+byte_to_char[rb]
            ids.append(tokenizer.stoi.get(sym,unk)); offsets.append((cs,ce)); exact.append(ok); sb=eb
    return ids,offsets,exact

probe="IN THE SUPREME COURT OF INDIA\\nSection 14: ₹100 — legal test."
pids,poff,pexact=encode_with_char_offsets(probe)
assert tokenizer.decode(pids)==probe
print("Round-trip exact: True")
print("Probe exact-boundary tokens:",sum(pexact),"/",len(pexact))


Tokenizer vocab: 24576
Token unit regex: \w+|[^\w\s]|\s+
Round-trip exact: True
Probe exact-boundary tokens: 27 / 27


In [23]:
PHASE2_FILES={
    "fold1_text":"Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold":"Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text":"Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold":"Phase_2_LNER/Train/fold2-gold.json",
    "labels":"Phase_2_LNER/Train/labels.txt",
    "test":"Phase_2_LNER/Test/fold3-text.json",
}
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        print("✅ Using HF_TOKEN from Kaggle Secrets (token value is not printed).")
except Exception:
    print("ℹ️ HF_TOKEN secret not available; trying public Hugging Face access.")

def get_hf_file(filename):
    return hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename=filename,
        token=HF_TOKEN if HF_TOKEN else None,
    )
phase2_paths={k:get_hf_file(v) for k,v in PHASE2_FILES.items()}
for k,v in phase2_paths.items(): print(f"{k:12s} -> {v}")

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

with open(phase2_paths["labels"],encoding="utf-8") as f:
    LABELS=[x.strip() for x in f if x.strip()]
assert len(LABELS)==12 and len(set(LABELS))==12,LABELS
print("Official labels:",LABELS)


✅ Using HF_TOKEN from Kaggle Secrets (token value is not printed).
fold1_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json
fold1_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json
fold2_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json
fold2_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json
labels       -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt
test         -> /root/.cache/huggingface/hub/datasets--Exploration-L

In [24]:
def normalize_text_json(obj):
    if isinstance(obj,dict):
        if all(isinstance(v,str) for v in obj.values()): return {str(k):v for k,v in obj.items()}
        if "data" in obj: return normalize_text_json(obj["data"])
    if isinstance(obj,list):
        out={}
        for r in obj:
            did=r.get("id",r.get("doc_id",r.get("document_id")))
            txt=r.get("text",r.get("document"))
            if did is None or not isinstance(txt,str): raise ValueError(f"Cannot parse text record: {r}")
            out[str(did)]=txt
        return out
    raise TypeError(f"Unsupported text JSON type: {type(obj)}")

def normalize_one_entity(e):
    if isinstance(e,list):
        if len(e) not in (3,4): raise ValueError(f"Bad entity: {e}")
        a,b,l=e[:3]; et=e[3] if len(e)==4 else None
        return int(a),int(b),str(l),et
    if isinstance(e,dict):
        a=e.get("start",e.get("start_char",e.get("start_character")))
        b=e.get("end",e.get("end_char",e.get("end_character")))
        l=e.get("label",e.get("type",e.get("entity_type")))
        et=e.get("text",e.get("entity_text"))
        if a is None or b is None or l is None: raise ValueError(f"Cannot parse entity: {e}")
        return int(a),int(b),str(l),et
    raise TypeError(type(e))

def normalize_gold_json(obj):
    if isinstance(obj,dict) and "data" in obj: return normalize_gold_json(obj["data"])
    if isinstance(obj,dict):
        out={}
        for did,ents in obj.items():
            if isinstance(ents,dict): ents=ents.get("entities",ents.get("annotations",ents.get("spans")))
            if ents is None: ents=[]
            out[str(did)]=[normalize_one_entity(e) for e in ents]
        return out
    if isinstance(obj,list):
        out={}
        for r in obj:
            did=r.get("id",r.get("doc_id",r.get("document_id")))
            ents=r.get("entities",r.get("annotations",r.get("spans")))
            if did is None or ents is None: raise ValueError(f"Cannot parse gold record: {r}")
            out[str(did)]=[normalize_one_entity(e) for e in ents]
        return out
    raise TypeError(type(obj))

fold1_text=normalize_text_json(read_json(phase2_paths["fold1_text"]))
fold2_text=normalize_text_json(read_json(phase2_paths["fold2_text"]))
test_text=normalize_text_json(read_json(phase2_paths["test"]))
fold1_gold=normalize_gold_json(read_json(phase2_paths["fold1_gold"]))
fold2_gold=normalize_gold_json(read_json(phase2_paths["fold2_gold"]))
assert set(fold1_text)==set(fold1_gold)
assert set(fold2_text)==set(fold2_gold)

allowed=set(LABELS)
def validate_gold(text_map,gold_map,name):
    n_ent=0; bad_text=0; overlaps=0; counts=Counter(); lengths=[]
    for did,text in text_map.items():
        prev=[]
        for a,b,l,et in gold_map[did]:
            assert 0<=a<b<=len(text),(did,a,b,len(text))
            assert l in allowed,(did,l)
            if et is not None and text[a:b]!=et: bad_text+=1
            for aa,bb,ll,_ in prev:
                if max(a,aa)<min(b,bb): overlaps+=1
            prev.append((a,b,l,et)); n_ent+=1; counts[l]+=1; lengths.append(b-a)
    print(f"{name}: docs={len(text_map):,} entities={n_ent:,} entity_text_mismatch={bad_text} overlapping_pairs={overlaps}")
    print(" label counts:",dict(counts))
    if lengths:
        print(" entity char length p50/p90/max:",int(np.percentile(lengths,50)),int(np.percentile(lengths,90)),max(lengths))

validate_gold(fold1_text,fold1_gold,"Fold1")
validate_gold(fold2_text,fold2_gold,"Fold2")
print("Test docs:",len(test_text))


Fold1: docs=35 entities=4,120 entity_text_mismatch=0 overlapping_pairs=0
 label counts: {'RESP': 191, 'DATE': 813, 'COURT': 455, 'WIT': 67, 'APP': 207, 'STAT': 525, 'A.COUNSEL': 70, 'PREC': 624, 'JUDGE': 123, 'R.COUNSEL': 87, 'CASENO': 423, 'AUTH': 535}
 entity char length p50/p90/max: 13 34 195
Fold2: docs=35 entities=4,273 entity_text_mismatch=0 overlapping_pairs=0
 label counts: {'JUDGE': 122, 'CASENO': 337, 'APP': 200, 'RESP': 190, 'A.COUNSEL': 125, 'R.COUNSEL': 97, 'STAT': 735, 'DATE': 762, 'AUTH': 468, 'PREC': 559, 'COURT': 558, 'WIT': 120}
 entity char length p50/p90/max: 13 32 233
Test docs: 35


In [25]:
TAG_NAMES=["O"]
for lab in LABELS: TAG_NAMES += [f"B-{lab}",f"I-{lab}",f"L-{lab}",f"U-{lab}"]
TAG2ID={t:i for i,t in enumerate(TAG_NAMES)}
ID2TAG={i:t for t,i in TAG2ID.items()}
O_ID=TAG2ID["O"]; IGNORE_INDEX=-100; N_TAGS=len(TAG_NAMES)

def tag_document(text,entities):
    ids,offsets,exact=encode_with_char_offsets(text)
    tags=[O_ID]*len(ids); group=[-1]*len(ids)
    start_to_idx={}; end_to_idx={}
    for i,((a,b),ex) in enumerate(zip(offsets,exact)):
        if ex:
            start_to_idx.setdefault(a,i); end_to_idx[b]=i
    stats=Counter(); ranges=[]
    for gi,(a,b,lab,et) in enumerate(sorted(entities,key=lambda z:(z[0],z[1],z[2]))):
        if a not in start_to_idx or b not in end_to_idx:
            stats["unaligned"]+=1; continue
        i=start_to_idx[a]; j=end_to_idx[b]
        if j<i or offsets[i][0]!=a or offsets[j][1]!=b:
            stats["unaligned"]+=1; continue
        if not all(offsets[k][1]==offsets[k+1][0] for k in range(i,j)):
            stats["unaligned"]+=1; continue
        if any(group[k]!=-1 for k in range(i,j+1)):
            stats["overlap_conflict"]+=1; continue
        if i==j:
            tags[i]=TAG2ID[f"U-{lab}"]
        else:
            tags[i]=TAG2ID[f"B-{lab}"]
            for k in range(i+1,j): tags[k]=TAG2ID[f"I-{lab}"]
            tags[j]=TAG2ID[f"L-{lab}"]
        for k in range(i,j+1): group[k]=gi
        ranges.append((i,j,gi)); stats["aligned"]+=1
    return ids,offsets,tags,group,ranges,stats

def alignment_report(text_map,gold_map,name):
    total=Counter(); tok_lens=[]
    for did,text in text_map.items():
        ids,offsets,tags,group,ranges,st=tag_document(text,gold_map[did])
        total.update(st); tok_lens.append(len(ids))
    denom=total["aligned"]+total["unaligned"]+total["overlap_conflict"]
    print(f"{name} alignment:",dict(total))
    print("Representable gold spans:",f"{total['aligned']}/{denom} = {total['aligned']/max(1,denom):.4%}")
    print("Token lengths p50/p90/p95/max:",[int(np.percentile(tok_lens,q)) for q in (50,90,95)],max(tok_lens))

alignment_report(fold1_text,fold1_gold,"Fold1")
alignment_report(fold2_text,fold2_gold,"Fold2")


Fold1 alignment: {'aligned': 4093, 'unaligned': 27}
Representable gold spans: 4093/4120 = 99.3447%
Token lengths p50/p90/p95/max: [10181, 19167, 21017] 40485
Fold2 alignment: {'aligned': 4257, 'unaligned': 16}
Representable gold spans: 4257/4273 = 99.6256%
Token lengths p50/p90/p95/max: [10734, 16185, 24784] 44587


In [26]:
import bisect

class NERWindowDataset(Dataset):
    def __init__(self,text_map,gold_map=None,seq_len=NER_SEQ_LEN,step=WINDOW_STEP):
        self.text_map=text_map; self.gold_map=gold_map; self.seq_len=seq_len; self.step=step
        self.docs={}; self.windows=[]
        t0=time.time()
        for di,(did,text) in enumerate(text_map.items()):
            if gold_map is None:
                ids,offsets,exact=encode_with_char_offsets(text)
                tags=None; group=None; ranges=[]; stats=Counter()
            else:
                ids,offsets,tags,group,ranges,stats=tag_document(text,gold_map[did])
            ranges=sorted(ranges)                                   # (first_tok, last_tok, gold_idx), non-overlapping
            self.docs[did]=dict(text=text,ids=ids,offsets=offsets,tags=tags,group=group,ranges=ranges,
                                r_last=[r[1] for r in ranges],stats=stats)
            n=len(ids)
            if n==0: self.windows.append((did,0,0))
            else:
                for s in range(0,max(1,n),step):
                    e=min(n,s+seq_len); self.windows.append((did,s,e))
                    if e==n: break
            if (di+1)%100==0: print(f" tokenized {di+1}/{len(text_map)} docs",flush=True)
        print(f"Built {len(self.windows):,} windows from {len(self.docs):,} docs in {(time.time()-t0)/60:.1f} min")
    def __len__(self): return len(self.windows)
    def __getitem__(self,idx):
        did,s,e=self.windows[idx]; d=self.docs[did]; ids=d["ids"][s:e]
        item={"doc_id":did,"start_token":s,"end_token":e,"input_ids":torch.tensor(ids,dtype=torch.long)}
        if d["tags"] is not None:
            labels=list(d["tags"][s:e])
            # entities that are cut by the window border are ignored (no loss on their visible fragment)
            k=bisect.bisect_left(d["r_last"],s)
            while k<len(d["ranges"]):
                i,j,_=d["ranges"][k]
                if i>=e: break
                if i<s or j>=e:
                    for p in range(max(i,s),min(j,e-1)+1): labels[p-s]=IGNORE_INDEX
                k+=1
            item["labels"]=torch.tensor(labels,dtype=torch.long)
        return item

def collate_windows(batch):
    B=len(batch); T=max(max(1,len(x["input_ids"])) for x in batch)
    input_ids=torch.full((B,T),tokenizer.pad_token_id,dtype=torch.long)
    attention_mask=torch.zeros((B,T),dtype=torch.bool)
    labels=torch.full((B,T),IGNORE_INDEX,dtype=torch.long) if "labels" in batch[0] else None
    doc_ids=[]; starts=[]; ends=[]
    for i,x in enumerate(batch):
        n=len(x["input_ids"])
        if n:
            input_ids[i,:n]=x["input_ids"]; attention_mask[i,:n]=True
            if labels is not None: labels[i,:n]=x["labels"]
        doc_ids.append(x["doc_id"]); starts.append(x["start_token"]); ends.append(x["end_token"])
    out={"input_ids":input_ids,"attention_mask":attention_mask,"doc_ids":doc_ids,"start_tokens":starts,"end_tokens":ends}
    if labels is not None: out["labels"]=labels
    return out

train_ds=NERWindowDataset(fold1_text,fold1_gold)
val_ds=NERWindowDataset(fold2_text,fold2_gold)
print("Train windows:",len(train_ds))
print("Val windows:",len(val_ds))


Built 736 windows from 35 docs in 0.0 min
Built 748 windows from 35 docs in 0.0 min
Train windows: 736
Val windows: 748


In [27]:
class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-5):
        super().__init__(); self.weight=nn.Parameter(torch.ones(dim)); self.eps=eps
    def forward(self,x):
        dtype=x.dtype; x32=x.float()
        x32=x32*torch.rsqrt(x32.pow(2).mean(-1,keepdim=True)+self.eps)
        return x32.to(dtype)*self.weight

class RotaryTable(nn.Module):
    def __init__(self,head_dim,max_positions,base=10000.0):
        super().__init__(); assert head_dim%2==0
        inv=1.0/(base**(torch.arange(0,head_dim,2,dtype=torch.float32)/head_dim))
        freqs=torch.outer(torch.arange(max_positions,dtype=torch.float32),inv)
        self.register_buffer("cos_cached",freqs.cos(),persistent=False)
        self.register_buffer("sin_cached",freqs.sin(),persistent=False)
        self.max_positions=max_positions
    def forward(self,position_ids):
        return self.cos_cached[position_ids].unsqueeze(1),self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x,cos,sin):
    B,H,T,D=x.shape; cos=cos.to(x.dtype); sin=sin.to(x.dtype)
    xp=x.view(B,H,T,D//2,2); x0,x1=xp[...,0],xp[...,1]
    return torch.stack((x0*cos-x1*sin,x0*sin+x1*cos),dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH=True
    BIDIRECTIONAL=False   # set True for NER (no causal mask; padding mask only)
    def __init__(self,d_model,n_heads,n_kv_heads,dropout):
        super().__init__(); assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads; self.n_kv_heads=n_kv_heads; self.head_dim=d_model//n_heads
        self.kv_repeat=n_heads//n_kv_heads; self.dropout=dropout
        self.q_proj=nn.Linear(d_model,n_heads*self.head_dim,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.o_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x,cos,sin,past_kv=None,use_cache=False,key_valid=None):
        B,T,C=x.shape; H,KV,D=self.n_heads,self.n_kv_heads,self.head_dim
        q=self.q_proj(x).view(B,T,H,D).transpose(1,2)
        k=self.k_proj(x).view(B,T,KV,D).transpose(1,2)
        v=self.v_proj(x).view(B,T,KV,D).transpose(1,2)
        q=apply_rope(q,cos,sin); k=apply_rope(k,cos,sin)
        past_len=0
        if past_kv is not None:
            pk,pv=past_kv; past_len=pk.size(2); k=torch.cat([pk,k],2); v=torch.cat([pv,v],2)
        new_kv=(k,v) if use_cache else None; S=k.size(2); drop=self.dropout if self.training else 0.0
        if T==1 and past_len>0 and self.GQA_DECODE_PATH:
            qg=q.reshape(B,KV,self.kv_repeat,D); mask=None if key_valid is None else key_valid[:,None,None,:]
            y=F.scaled_dot_product_attention(qg,k,v,attn_mask=mask,dropout_p=drop,is_causal=False)
            y=y.reshape(B,H,1,D).transpose(1,2).contiguous().view(B,1,C)
            return self.o_proj(y),new_kv
        if self.kv_repeat!=1:
            k_attn=k.repeat_interleave(self.kv_repeat,1); v_attn=v.repeat_interleave(self.kv_repeat,1)
        else: k_attn,v_attn=k,v
        attn_mask,is_causal=None,False
        if self.BIDIRECTIONAL:
            if key_valid is not None: attn_mask=key_valid[:,None,None,:]
        elif T==1:
            if key_valid is not None: attn_mask=key_valid[:,None,None,:]
        elif key_valid is None and past_len==0:
            is_causal=True
        else:
            qi=torch.arange(T,device=x.device)[:,None]+past_len
            kj=torch.arange(S,device=x.device)[None,:]
            attn_mask=(kj<=qi)[None,None,:,:]
            if key_valid is not None:
                attn_mask=attn_mask & key_valid[:,None,None,:]
                if past_len==0: attn_mask=attn_mask | (kj==qi)[None,None,:,:]
        y=F.scaled_dot_product_attention(q,k_attn,v_attn,attn_mask=attn_mask,dropout_p=drop,is_causal=is_causal)
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.o_proj(y),new_kv

class SwiGLU(nn.Module):
    def __init__(self,d_model,hidden_dim,dropout):
        super().__init__(); self.gate_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.up_proj=nn.Linear(d_model,hidden_dim,bias=False); self.down_proj=nn.Linear(hidden_dim,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x): return self.dropout(self.down_proj(F.silu(self.gate_proj(x))*self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,n_kv_heads,ffn_dim,dropout,rms_eps):
        super().__init__(); self.norm1=RMSNorm(d_model,rms_eps)
        self.attn=GQACausalSelfAttention(d_model,n_heads,n_kv_heads,dropout)
        self.norm2=RMSNorm(d_model,rms_eps); self.ff=SwiGLU(d_model,ffn_dim,dropout); self.resid_drop=nn.Dropout(dropout)
    def forward(self,x,cos,sin,past_kv=None,use_cache=False,key_valid=None):
        a,new_kv=self.attn(self.norm1(x),cos,sin,past_kv=past_kv,use_cache=use_cache,key_valid=key_valid)
        x=x+self.resid_drop(a); x=x+self.ff(self.norm2(x)); return x,new_kv

class EfficientDecoderLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,n_kv_heads,ffn_dim,dropout,max_positions,rope_base,rms_eps):
        super().__init__(); self.vocab_size=vocab_size; self.d_model=d_model; self.max_positions=max_positions
        self.token_emb=nn.Embedding(vocab_size,d_model); self.drop=nn.Dropout(dropout)
        self.rope=RotaryTable(d_model//n_heads,max_positions,rope_base)
        self.blocks=nn.ModuleList([EfficientDecoderBlock(d_model,n_heads,n_kv_heads,ffn_dim,dropout,rms_eps) for _ in range(n_layers)])
        self.final_norm=RMSNorm(d_model,rms_eps)
    def forward_hidden(self,input_ids,past_key_values=None,use_cache=False,position_ids=None,key_valid=None):
        B,T=input_ids.shape; past_len=0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len+T>self.max_positions: raise ValueError(f"context {past_len+T} exceeds max_positions={self.max_positions}")
        if position_ids is None:
            position_ids=torch.arange(past_len,past_len+T,device=input_ids.device)[None,:].expand(B,T)
        cos,sin=self.rope(position_ids); x=self.drop(self.token_emb(input_ids)); new_cache=[] if use_cache else None
        for i,block in enumerate(self.blocks):
            past=None if past_key_values is None else past_key_values[i]
            x,kv=block(x,cos,sin,past_kv=past,use_cache=use_cache,key_valid=key_valid)
            if use_cache: new_cache.append(kv)
        return self.final_norm(x),new_cache
    def project(self,hidden): return F.linear(hidden,self.token_emb.weight)

class Model4ForNER(nn.Module):
    def __init__(self,backbone,n_tags,dropout=0.1):
        super().__init__(); self.backbone=backbone; self.ner_dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(backbone.d_model,n_tags)
    def forward(self,input_ids,attention_mask=None):
        h,_=self.backbone.forward_hidden(input_ids,key_valid=attention_mask)
        return self.classifier(self.ner_dropout(h))


In [28]:
phase1_obj=torch.load(MODEL4_BEST_PATH,map_location="cpu",weights_only=False)
cfg=phase1_obj["config"]
print("Step:",phase1_obj.get("global_step"),"val_loss:",phase1_obj.get("val_loss"))
print("Architecture:",cfg.get("architecture"))
assert cfg.get("architecture")=="efficient_gqa_rope_swiglu_decoder"
assert int(cfg["vocab_size"])==len(tokenizer)

backbone=EfficientDecoderLM(
    vocab_size=int(cfg["vocab_size"]),d_model=int(cfg["d_model"]),n_layers=int(cfg["n_layers"]),
    n_heads=int(cfg["n_heads"]),n_kv_heads=int(cfg["n_kv_heads"]),ffn_dim=int(cfg["ffn_dim"]),
    dropout=float(cfg.get("dropout",0.0)),max_positions=int(cfg["max_positions"]),
    rope_base=float(cfg.get("rope_base",10000.0)),rms_eps=float(cfg.get("rms_eps",1e-5))
)
backbone.load_state_dict(phase1_obj["model"],strict=True)
print("✅ Phase-1 Model 4 loaded STRICTLY")

GQACausalSelfAttention.BIDIRECTIONAL=bool(BIDIRECTIONAL_ATTENTION)
print("Bidirectional attention:",GQACausalSelfAttention.BIDIRECTIONAL)
model=Model4ForNER(backbone,N_TAGS,HEAD_DROPOUT).to(DEVICE)
total=sum(p.numel() for p in model.parameters()); trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total/1e6:.3f}M | trainable: {trainable/1e6:.3f}M")
assert total==trainable


Step: 16266 val_loss: 1.3664915735571412
Architecture: efficient_gqa_rope_swiglu_decoder
✅ Phase-1 Model 4 loaded STRICTLY
Bidirectional attention: True
Total params: 99.341M | trainable: 99.341M


In [29]:
# ===== TOKENIZER SANITY CHECK (run before training!) =====
# Heuristic compatibility check only. Phase-1 Model 4 val loss was ~1.37 on its own validation distribution.
# Phase-2 loss can differ because the data distribution is different. The SHA check above is the stronger tokenizer identity test.
# A very large loss (e.g. >6) is a red flag for tokenizer/checkpoint incompatibility or a model-definition mismatch.
@torch.no_grad()
def lm_loss_check(ds,n_windows=TOKENIZER_CHECK_WINDOWS):
    prev=GQACausalSelfAttention.BIDIRECTIONAL; GQACausalSelfAttention.BIDIRECTIONAL=False   # LM is causal
    backbone.to(DEVICE).eval(); losses=[]
    step=max(1,len(ds)//n_windows)
    for k in list(range(0,len(ds),step))[:n_windows]:
        ids=ds[k]["input_ids"][None,:].to(DEVICE)
        if ids.size(1)<32: continue
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
            h,_=backbone.forward_hidden(ids); lg=backbone.project(h)
        l=F.cross_entropy(lg[0,:-1].float(),ids[0,1:],reduction="mean")
        losses.append(float(l))
    GQACausalSelfAttention.BIDIRECTIONAL=prev
    return float(np.mean(losses))
chk=lm_loss_check(train_ds)
print(f"Model-4 causal LM loss on Phase-2 train windows: {chk:.3f}")
if chk>6: print("❌ Very high LM loss — stop and inspect tokenizer/checkpoint/model-definition compatibility before training.")
elif chk>4: print("⚠️ Higher than expected; check the tokenizer / checkpoint before training.")
else: print("✅ Tokenizer + checkpoint look consistent.")


Model-4 causal LM loss on Phase-2 train windows: 2.029
✅ Tokenizer + checkpoint look consistent.


In [30]:
def count_training_tags(ds):
    c=Counter()
    for d in ds.docs.values():
        if d["tags"] is not None: c.update(d["tags"])
    return c

tag_counts=count_training_tags(train_ds)
if USE_CLASS_WEIGHTS:
    counts=np.array([max(1,tag_counts.get(i,0)) for i in range(N_TAGS)],dtype=np.float64)
    weights=np.sqrt(counts.sum()/counts)
    weights/=np.median(weights[1:]) if len(weights)>1 else 1.0
    weights=np.clip(weights,0.20,5.0); weights[O_ID]=min(weights[O_ID],0.35)
    CLASS_WEIGHTS=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
else: CLASS_WEIGHTS=None

print("\nTraining tag statistics:")
for tid in range(N_TAGS):
    if tag_counts.get(tid,0)>0:
        w = float(CLASS_WEIGHTS[tid].detach().cpu()) if CLASS_WEIGHTS is not None else 1.0
        print(f"{ID2TAG[tid]:16s} count={tag_counts.get(tid,0):8d} weight={w:.3f}")

train_loader=DataLoader(train_ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True,collate_fn=collate_windows,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate_windows,num_workers=2,pin_memory=True)

optimizer=torch.optim.AdamW([
    {"params":list(model.backbone.parameters()),"lr":BACKBONE_LR},
    {"params":list(model.classifier.parameters()),"lr":HEAD_LR},
],weight_decay=WEIGHT_DECAY)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=max(1,updates_per_epoch*EPOCHS); warmup_updates=max(1,int(WARMUP_RATIO*total_updates))
def lr_lambda(step):
    if step<warmup_updates: return (step+1)/warmup_updates
    progress=(step-warmup_updates)/max(1,total_updates-warmup_updates)
    return 0.5*(1+math.cos(math.pi*min(1.0,progress)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
scaler=torch.amp.GradScaler("cuda",enabled=(AMP and DEVICE=="cuda"))
print("Batches/epoch:",len(train_loader),"updates/epoch:",updates_per_epoch,"total updates:",total_updates)



Training tag statistics:
O                count=  390766 weight=0.200
B-APP            count=     169 weight=1.015
I-APP            count=     536 weight=0.570
L-APP            count=     169 weight=1.015
U-APP            count=      34 weight=2.262
B-RESP           count=     174 weight=1.000
I-RESP           count=     498 weight=0.591
L-RESP           count=     174 weight=1.000
U-RESP           count=      17 weight=3.199
B-A.COUNSEL      count=      64 weight=1.649
I-A.COUNSEL      count=     275 weight=0.795
L-A.COUNSEL      count=      64 weight=1.649
U-A.COUNSEL      count=       6 weight=5.000
B-R.COUNSEL      count=      83 weight=1.448
I-R.COUNSEL      count=     295 weight=0.768
L-R.COUNSEL      count=      83 weight=1.448
U-R.COUNSEL      count=       4 weight=5.000
B-JUDGE          count=     118 weight=1.214
I-JUDGE          count=     574 weight=0.551
L-JUDGE          count=     118 weight=1.214
U-JUDGE          count=       4 weight=5.000
B-WIT            count=      

In [31]:
def strict_scores(truth,pred,labels=LABELS,verbose=True):
    if set(truth)!=set(pred): raise ValueError("ID mismatch")
    stats={l:[0,0,0] for l in labels}; allowed=set(labels)
    def ss(xs):
        out=set()
        for s in xs or []:
            a,b,l=s[:3]
            if not isinstance(a,int) or isinstance(a,bool) or not isinstance(b,int) or isinstance(b,bool): raise ValueError("bad offsets")
            if a<0 or b<=a or l not in allowed: raise ValueError("bad span")
            t=(a,b,l)
            if t in out: raise ValueError("duplicate span")
            out.add(t)
        return out
    for did in truth:
        g=ss(truth[did]); p=ss(pred[did])
        for l in labels:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    ps=[]; rs=[]; fs=[]; per={}
    for l in labels:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        f=2*pr*rc/(pr+rc) if pr+rc else 0
        ps.append(pr); rs.append(rc); fs.append(f); per[l]=(pr,rc,f,tp,fp,fn)
        if verbose: print(f"{l:10s}: P={pr:.6f} R={rc:.6f} F1={f:.6f}")
    out={"macro_f1":float(np.mean(fs)),"macro_precision":float(np.mean(ps)),"macro_recall":float(np.mean(rs)),"per_label":per}
    if verbose:
        print(f"Strict Macro-F1:        {out['macro_f1']:.6f}")
        print(f"Strict Macro-Precision: {out['macro_precision']:.6f}")
        print(f"Strict Macro-Recall:    {out['macro_recall']:.6f}")
    return out

def gold_for_eval(gold_map):
    out={}
    for did,ents in gold_map.items():
        seen=set(); lst=[]
        for a,b,l,et in ents:
            t=(int(a),int(b),str(l))
            if t not in seen: seen.add(t); lst.append(list(t))      # evaluator rejects duplicate spans
        out[did]=lst
    return out

# ---- constrained BILOU Viterbi ----
def _build_transitions():
    pref=[None]+[t.split("-",1)[0] for t in TAG_NAMES[1:]]
    lab=[None]+[t.split("-",1)[1] for t in TAG_NAMES[1:]]
    allow=np.zeros((N_TAGS,N_TAGS),dtype=bool)
    for a in range(N_TAGS):
        a_open = pref[a] in ("B","I")
        for b in range(N_TAGS):
            if a_open: allow[a,b]=(pref[b] in ("I","L") and lab[b]==lab[a])
            else:      allow[a,b]=(b==O_ID or pref[b] in ("B","U"))
    start=np.array([ (i==O_ID or pref[i] in ("B","U")) for i in range(N_TAGS)])
    end=np.array([ (i==O_ID or pref[i] in ("L","U")) for i in range(N_TAGS)])
    return allow,start,end
TRANS_ALLOW,START_OK,END_OK=_build_transitions()
NEG=-1e9

def softmax_np(x):
    x=x-x.max(-1,keepdims=True); e=np.exp(x); return e/e.sum(-1,keepdims=True)

def viterbi_path(logp):
    n=logp.shape[0]
    trans=np.where(TRANS_ALLOW,0.0,NEG).astype(np.float32)
    score=np.where(START_OK,logp[0],NEG).astype(np.float32)
    back=np.zeros((n,N_TAGS),dtype=np.int16)
    for t in range(1,n):
        cand=score[:,None]+trans                     # (from,to)
        bp=cand.argmax(0); score=cand[bp,np.arange(N_TAGS)]+logp[t]; back[t]=bp
    score=np.where(END_OK,score,NEG)
    path=np.zeros(n,dtype=np.int64); path[-1]=int(score.argmax())
    for t in range(n-1,0,-1): path[t-1]=back[t,path[t]]
    return path

def decode_spans(doc,logits,viterbi=None):
    """Returns [(start_char,end_char,label,confidence)], confidence = min token prob of the span's tags."""
    if len(logits)==0: return []
    viterbi=USE_VITERBI if viterbi is None else viterbi
    logits=np.asarray(logits,dtype=np.float32); probs=softmax_np(logits)
    if viterbi: pred=viterbi_path(np.log(probs+1e-9))
    else:       pred=probs.argmax(-1)
    conf=probs[np.arange(len(pred)),pred]; offsets=doc["offsets"]; text=doc["text"]; out=[]; i=0; n=len(pred)
    while i<n:
        tag=ID2TAG[int(pred[i])]
        if tag=="O": i+=1; continue
        pf,lab=tag.split("-",1)
        if pf=="U": j=i
        elif pf=="B":
            j=i+1
            while j<n and ID2TAG[int(pred[j])]==f"I-{lab}": j+=1
            if not(j<n and ID2TAG[int(pred[j])]==f"L-{lab}"): i+=1; continue     # malformed (argmax mode only)
        else: i+=1; continue
        a=offsets[i][0]; b=offsets[j][1]
        if 0<=a<b<=len(text) and text[a:b].strip(): out.append((int(a),int(b),lab,float(conf[i:j+1].min())))
        i=j+1
    return out

def _threshold_for_label(threshold,label):
    if isinstance(threshold,dict):
        return float(threshold.get(label,0.0))
    return float(threshold)

def filter_spans(spans,threshold=0.0):
    """threshold may be a scalar or {label: threshold} dict."""
    seen=set(); res=[]
    for a,b,l,c in spans:
        th=_threshold_for_label(threshold,l)
        if c>=th and (a,b,l) not in seen:
            seen.add((a,b,l)); res.append([a,b,l])
    return res

def decode_bilou_from_logits(doc,logits,threshold=0.0,viterbi=None):
    return filter_spans(decode_spans(doc,logits,viterbi),threshold)

def tune_global_threshold(span_cache,truth,grid):
    rows=[]
    for th in grid:
        pred={did:filter_spans(span_cache[did],th) for did in span_cache}
        sc=strict_scores(truth,pred,verbose=False)
        rows.append((sc["macro_f1"],float(th),sc))
    rows.sort(reverse=True,key=lambda x:x[0])
    return rows[0],rows

def tune_per_label_thresholds(span_cache,truth,grid):
    """
    Macro-F1 is the unweighted mean of per-label F1 values.
    For a fixed decoded span set, each label's confidence threshold affects only
    that label's F1, so independent per-label tuning directly optimizes this
    held-out Macro-F1 threshold family.
    """
    thresholds={}
    details={}
    for lab in LABELS:
        best=(-1.0,0.0,None)
        for th in grid:
            pred={}
            for did,spans in span_cache.items():
                pred[did]=[[a,b,l] for a,b,l,c in spans if l==lab and c>=float(th)]
            # Score only this label using exact strict matching.
            sc=strict_scores(
                {did:[s for s in truth[did] if s[2]==lab] for did in truth},
                pred,
                labels=[lab],
                verbose=False,
            )
            row=(sc["macro_f1"],float(th),sc)
            if row[0]>best[0]:
                best=row
        thresholds[lab]=best[1]
        details[lab]=best[2]

    combined={did:filter_spans(span_cache[did],thresholds) for did in span_cache}
    combined_sc=strict_scores(truth,combined,verbose=False)
    return thresholds,combined_sc,details


In [32]:
# ===== ORACLE CEILING TEST: gold tags -> decode -> strict F1 (no model involved) =====
# This is the best Strict Macro-F1 your tokenizer/alignment can possibly give on Fold 2.
# If it is far below ~0.95, token boundaries do not line up with gold spans (try PRESPLIT_PUNCT=True and compare).
def oracle_logits(doc):
    lg=np.full((len(doc["ids"]),N_TAGS),-10.0,dtype=np.float32)
    for k,t in enumerate(doc["tags"]): lg[k,t]=10.0
    return lg
oracle_pred={did:decode_bilou_from_logits(d,oracle_logits(d),0.0) for did,d in val_ds.docs.items()}
_truth_val=gold_for_eval(fold2_gold)
print("ORACLE (Fold 2):"); oracle_sc=strict_scores(_truth_val,oracle_pred,verbose=True)


ORACLE (Fold 2):
APP       : P=1.000000 R=0.995000 F1=0.997494
RESP      : P=1.000000 R=1.000000 F1=1.000000
A.COUNSEL : P=1.000000 R=1.000000 F1=1.000000
R.COUNSEL : P=1.000000 R=1.000000 F1=1.000000
JUDGE     : P=1.000000 R=1.000000 F1=1.000000
WIT       : P=1.000000 R=1.000000 F1=1.000000
AUTH      : P=1.000000 R=1.000000 F1=1.000000
COURT     : P=1.000000 R=0.980287 F1=0.990045
STAT      : P=1.000000 R=0.994558 F1=0.997271
PREC      : P=1.000000 R=1.000000 F1=1.000000
DATE      : P=1.000000 R=1.000000 F1=1.000000
CASENO    : P=1.000000 R=1.000000 F1=1.000000
Strict Macro-F1:        0.998734
Strict Macro-Precision: 1.000000
Strict Macro-Recall:    0.997487


In [33]:
@torch.no_grad()
def infer_dataset(model,ds,loader,threshold=0.0,return_logits=False):
    model.eval()
    doc_logits={did:np.zeros((len(d["ids"]),N_TAGS),dtype=np.float32) for did,d in ds.docs.items()}
    best_dist={did:np.full(len(d["ids"]),-1.0,dtype=np.float32) for did,d in ds.docs.items()}   # centrality of the chosen logits
    for bi,batch in enumerate(loader):
        ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
            logits=model(ids,mask)
        logits=logits.float().cpu().numpy()
        for r,did in enumerate(batch["doc_ids"]):
            s=batch["start_tokens"][r]; e=batch["end_tokens"][r]; n=len(ds.docs[did]["ids"])
            if e<=s: continue
            L=e-s; loc=np.arange(L)
            dl=loc if s>0 else np.full(L,1e9); dr=(L-1-loc) if e<n else np.full(L,1e9)   # no penalty at true doc borders
            dist=np.minimum(dl,dr).astype(np.float32)
            upd=dist>best_dist[did][s:e]
            doc_logits[did][s:e][upd]=logits[r,:L][upd]; best_dist[did][s:e][upd]=dist[upd]
        if (bi+1)%50==0: print(f"inference batch {bi+1}/{len(loader)}",flush=True)
    for did in best_dist:
        if len(best_dist[did]) and (best_dist[did]<0).any(): raise RuntimeError(f"{did}: unfilled tokens")
    pred={did:decode_bilou_from_logits(ds.docs[did],doc_logits[did],threshold) for did in ds.docs}
    return (pred,doc_logits) if return_logits else pred


In [34]:
def compute_loss(logits,labels):
    return F.cross_entropy(logits.float().view(-1,N_TAGS),labels.view(-1),weight=CLASS_WEIGHTS,ignore_index=IGNORE_INDEX)

best_f1=-1.0; best_epoch=None; global_update=0; history=[]
truth_val=gold_for_eval(fold2_gold)

for epoch in range(1,EPOCHS+1):
    model.train(); optimizer.zero_grad(set_to_none=True); running=0.0; n_batches=0; t0=time.time()
    for step,batch in enumerate(train_loader,1):
        ids=batch["input_ids"].to(DEVICE,non_blocking=True); mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
        y=batch["labels"].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
            logits=model(ids,mask); loss=compute_loss(logits,y)/GRAD_ACCUM_STEPS
        scaler.scale(loss).backward(); running+=float(loss.detach().cpu())*GRAD_ACCUM_STEPS; n_batches+=1
        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step(); global_update+=1
        if step%100==0 or step==len(train_loader):
            print(f"epoch {epoch}/{EPOCHS} step {step}/{len(train_loader)} loss={running/max(1,n_batches):.4f} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e} elapsed={(time.time()-t0)/60:.1f}m",flush=True)

    _,val_logits=infer_dataset(model,val_ds,val_loader,0.0,True)
    span_cache={did:decode_spans(val_ds.docs[did],val_logits[did]) for did in val_ds.docs}

    (global_f1,global_th,global_sc),_ = tune_global_threshold(span_cache,truth_val,THRESHOLD_GRID)

    if USE_PER_LABEL_THRESHOLDS:
        epoch_th,epoch_sc,_ = tune_per_label_thresholds(span_cache,truth_val,THRESHOLD_GRID)
        epoch_f1=epoch_sc["macro_f1"]
        threshold_desc="per-label"
    else:
        epoch_th=float(global_th); epoch_sc=global_sc; epoch_f1=global_f1
        threshold_desc=f"global={global_th:.2f}"

    print(
        f"Epoch {epoch}: global-threshold Macro-F1={global_f1:.6f} @ {global_th:.2f} | "
        f"selected({threshold_desc}) Macro-F1={epoch_f1:.6f} "
        f"P={epoch_sc['macro_precision']:.6f} R={epoch_sc['macro_recall']:.6f}"
    )
    if isinstance(epoch_th,dict):
        print("   per-label thresholds:"," ".join(f"{l}={epoch_th[l]:.2f}" for l in LABELS))

    history.append({
        "epoch":epoch,
        "train_loss":running/max(1,n_batches),
        "threshold":epoch_th,
        "global_threshold":float(global_th),
        "global_macro_f1":float(global_f1),
        "macro_f1":epoch_f1,
        "macro_precision":epoch_sc["macro_precision"],
        "macro_recall":epoch_sc["macro_recall"],
    })
    if epoch_f1>best_f1:
        best_f1=epoch_f1; best_epoch=epoch
        torch.save({"model":model.state_dict(),"phase1_checkpoint":MODEL4_BEST_PATH,"phase1_config":cfg,
                    "tag_names":TAG_NAMES,"labels":LABELS,"best_epoch":best_epoch,"best_threshold":epoch_th,
                    "strict_macro_f1":epoch_f1,"history":history,
                    "experiment":"Phase2 Exp1 Model4 full fine-tuning BILOU",
                    "ner_seq_len":NER_SEQ_LEN,"window_overlap":WINDOW_OVERLAP,
                    "bidirectional":bool(BIDIRECTIONAL_ATTENTION),"viterbi":bool(USE_VITERBI),"prespl_punct":bool(PRESPLIT_PUNCT),
                    "use_per_label_thresholds":bool(USE_PER_LABEL_THRESHOLDS),"threshold_grid":THRESHOLD_GRID,
                    "tokenizer_sha256":tok_sha},BEST_NER_PATH)
        print("✅ New best saved:",BEST_NER_PATH)
        print("   per-label F1:"," ".join(f"{l}={v[2]:.3f}" for l,v in epoch_sc["per_label"].items()))
    print("-"*90)

print("Best epoch:",best_epoch,"Best held-out Strict Macro-F1:",best_f1)
for x in history: print(x)


/tmp/ipykernel_49/902491516.py:17: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step(); global_update+=1


epoch 1/6 step 100/368 loss=3.7060 lr=2.41e-05 elapsed=0.2m
epoch 1/6 step 200/368 loss=2.2234 lr=4.81e-05 elapsed=0.4m
epoch 1/6 step 300/368 loss=1.6449 lr=4.98e-05 elapsed=0.6m
epoch 1/6 step 368/368 loss=1.4024 lr=4.93e-05 elapsed=0.8m
inference batch 50/187
inference batch 100/187
inference batch 150/187
Epoch 1: global-threshold Macro-F1=0.274775 @ 0.00 | selected(per-label) Macro-F1=0.281215 P=0.297829 R=0.276140
   per-label thresholds: APP=0.05 RESP=0.00 A.COUNSEL=0.00 R.COUNSEL=0.00 JUDGE=0.00 WIT=0.00 AUTH=0.05 COURT=0.45 STAT=0.05 PREC=0.00 DATE=0.10 CASENO=0.00
✅ New best saved: /kaggle/working/phase2_exp1_model4_fullft/best_ner.pt
   per-label F1: APP=0.059 RESP=0.000 A.COUNSEL=0.146 R.COUNSEL=0.142 JUDGE=0.118 WIT=0.000 AUTH=0.286 COURT=0.764 STAT=0.591 PREC=0.189 DATE=0.883 CASENO=0.198
------------------------------------------------------------------------------------------
epoch 2/6 step 100/368 loss=0.2147 lr=4.81e-05 elapsed=0.2m
epoch 2/6 step 200/368 loss=0.1948 

In [42]:
best_obj=torch.load(BEST_NER_PATH,map_location="cpu",weights_only=False)
model.load_state_dict(best_obj["model"],strict=True); model.to(DEVICE)
BEST_THRESHOLD=best_obj["best_threshold"]; BEST_EPOCH=int(best_obj["best_epoch"])
val_pred=infer_dataset(model,val_ds,val_loader,BEST_THRESHOLD)
val_score=strict_scores(truth_val,val_pred,verbose=True)
with open(VAL_PRED_PATH,"w",encoding="utf-8") as f: json.dump(val_pred,f,ensure_ascii=False,indent=2)
print("Best epoch:",BEST_EPOCH)
print("Best threshold(s):",BEST_THRESHOLD)
print("Saved:",VAL_PRED_PATH)


inference batch 50/187
inference batch 100/187
inference batch 150/187
APP       : P=0.214286 R=0.105000 F1=0.140940
RESP      : P=0.246575 R=0.094737 F1=0.136882
A.COUNSEL : P=0.475000 R=0.304000 F1=0.370732
R.COUNSEL : P=0.444444 R=0.371134 F1=0.404494
JUDGE     : P=0.628571 R=0.540984 F1=0.581498
WIT       : P=0.233333 R=0.058333 F1=0.093333
AUTH      : P=0.528205 R=0.220085 F1=0.310709
COURT     : P=0.913758 R=0.797491 F1=0.851675
STAT      : P=0.813839 R=0.672109 F1=0.736215
PREC      : P=0.464363 R=0.384615 F1=0.420744
DATE      : P=0.965706 R=0.923885 F1=0.944333
CASENO    : P=0.575630 R=0.406528 F1=0.476522
Strict Macro-F1:        0.455673
Strict Macro-Precision: 0.541976
Strict Macro-Recall:    0.406575
Best epoch: 5
Best threshold(s): {'APP': 0.0, 'RESP': 0.0, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.05, 'WIT': 0.0, 'AUTH': 0.1, 'COURT': 0.45, 'STAT': 0.05, 'PREC': 0.05, 'DATE': 0.05, 'CASENO': 0.05}
Saved: /kaggle/working/phase2_exp1_model4_fullft/val_prediction.json


In [43]:
import subprocess

cmd = [
    "python",
    "/kaggle/working/phase2_exp1_model4_fullft/evaluate_phase2.py",
    "--truth", phase2_paths["fold2_gold"],
    "--prediction", "/kaggle/working/phase2_exp1_model4_fullft/val_prediction.json",
    "--labels", phase2_paths["labels"],
]

print("Running official-style local evaluator...\n")

res = subprocess.run(
    cmd,
    text=True,
    capture_output=True
)

print(res.stdout)

if res.stderr:
    print("STDERR:")
    print(res.stderr)

print("Return code:", res.returncode)

Running official-style local evaluator...

Examples: 35
APP: P=0.214286 R=0.105000 F1=0.140940
RESP: P=0.246575 R=0.094737 F1=0.136882
A.COUNSEL: P=0.475000 R=0.304000 F1=0.370732
R.COUNSEL: P=0.444444 R=0.371134 F1=0.404494
JUDGE: P=0.628571 R=0.540984 F1=0.581498
WIT: P=0.233333 R=0.058333 F1=0.093333
AUTH: P=0.528205 R=0.220085 F1=0.310709
COURT: P=0.913758 R=0.797491 F1=0.851675
STAT: P=0.813839 R=0.672109 F1=0.736215
PREC: P=0.464363 R=0.384615 F1=0.420744
DATE: P=0.965706 R=0.923885 F1=0.944333
CASENO: P=0.575630 R=0.406528 F1=0.476522
Strict Macro-F1:        0.455673
Strict Macro-Precision: 0.541976
Strict Macro-Recall:    0.406575

Return code: 0


## Optional final refit on all public annotations
For model selection we kept Fold 2 held out. For the actual submission, you can refit a fresh copy of the Phase-1 Model 4 on **Fold 1 + Fold 2** for the selected number of epochs.

Set `DO_FINAL_REFIT_ON_ALL=True` in Cell 2, rerun Cell 2, then run the next cell.


In [38]:
FINAL_REFIT_PATH=ROOT/"final_refit_ner.pt"

def fresh_phase1_ner_model():
    bb=EfficientDecoderLM(vocab_size=int(cfg["vocab_size"]),d_model=int(cfg["d_model"]),n_layers=int(cfg["n_layers"]),
        n_heads=int(cfg["n_heads"]),n_kv_heads=int(cfg["n_kv_heads"]),ffn_dim=int(cfg["ffn_dim"]),
        dropout=float(cfg.get("dropout",0.0)),max_positions=int(cfg["max_positions"]),
        rope_base=float(cfg.get("rope_base",10000.0)),rms_eps=float(cfg.get("rms_eps",1e-5)))
    bb.load_state_dict(phase1_obj["model"],strict=True)
    return Model4ForNER(bb,N_TAGS,HEAD_DROPOUT).to(DEVICE)

if not DO_FINAL_REFIT_ON_ALL:
    print("Skipping final refit. Using held-out-selected best model for test inference.")
    final_model=model
else:
    all_text=dict(fold1_text); all_text.update(fold2_text)
    all_gold=dict(fold1_gold); all_gold.update(fold2_gold)
    all_ds=NERWindowDataset(all_text,all_gold)
    all_loader=DataLoader(all_ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True,collate_fn=collate_windows,num_workers=2,pin_memory=True)
    final_model=fresh_phase1_ner_model()

    all_counts=count_training_tags(all_ds)
    if USE_CLASS_WEIGHTS:
        counts=np.array([max(1,all_counts.get(i,0)) for i in range(N_TAGS)],dtype=np.float64)
        w=np.sqrt(counts.sum()/counts); w/=np.median(w[1:]); w=np.clip(w,0.20,5.0); w[O_ID]=min(w[O_ID],0.35)
        FW=torch.tensor(w,dtype=torch.float32,device=DEVICE)
    else: FW=None

    opt=torch.optim.AdamW([{"params":list(final_model.backbone.parameters()),"lr":BACKBONE_LR},
                           {"params":list(final_model.classifier.parameters()),"lr":HEAD_LR}],weight_decay=WEIGHT_DECAY)
    upe=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS); total=max(1,upe*BEST_EPOCH); warm=max(1,int(WARMUP_RATIO*total))
    def flr(step):
        if step<warm: return (step+1)/warm
        p=(step-warm)/max(1,total-warm); return 0.5*(1+math.cos(math.pi*min(1.0,p)))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,flr); scl=torch.amp.GradScaler("cuda",enabled=(AMP and DEVICE=="cuda"))

    for ep in range(1,BEST_EPOCH+1):
        final_model.train(); opt.zero_grad(set_to_none=True); run=0.0; t0=time.time()
        for step,batch in enumerate(all_loader,1):
            ids=batch["input_ids"].to(DEVICE,non_blocking=True); mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
            y=batch["labels"].to(DEVICE,non_blocking=True)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
                lg=final_model(ids,mask)
                loss=F.cross_entropy(lg.float().view(-1,N_TAGS),y.view(-1),weight=FW,ignore_index=IGNORE_INDEX)/GRAD_ACCUM_STEPS
            scl.scale(loss).backward(); run+=float(loss.detach().cpu())*GRAD_ACCUM_STEPS
            if step%GRAD_ACCUM_STEPS==0 or step==len(all_loader):
                scl.unscale_(opt); torch.nn.utils.clip_grad_norm_(final_model.parameters(),MAX_GRAD_NORM)
                scl.step(opt); scl.update(); opt.zero_grad(set_to_none=True); sched.step()
            if step%100==0 or step==len(all_loader):
                print(f"refit epoch {ep}/{BEST_EPOCH} step {step}/{len(all_loader)} loss={run/step:.4f} elapsed={(time.time()-t0)/60:.1f}m")
    torch.save({"model":final_model.state_dict(),"phase1_config":cfg,"tag_names":TAG_NAMES,"labels":LABELS,
                "epochs":BEST_EPOCH,"threshold":BEST_THRESHOLD,
                "experiment":"Phase2 Exp1 Model4 full fine-tuning BILOU final refit",
                "bidirectional":bool(BIDIRECTIONAL_ATTENTION),"viterbi":bool(USE_VITERBI),
                "use_per_label_thresholds":bool(USE_PER_LABEL_THRESHOLDS),
                "tokenizer_sha256":tok_sha},FINAL_REFIT_PATH)
    print("✅ Final refit saved:",FINAL_REFIT_PATH)


Built 1,484 windows from 70 docs in 0.0 min


/tmp/ipykernel_49/725681814.py:47: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scl.step(opt); scl.update(); opt.zero_grad(set_to_none=True); sched.step()


refit epoch 1/5 step 100/742 loss=3.7731 elapsed=0.2m
refit epoch 1/5 step 200/742 loss=2.4096 elapsed=0.4m
refit epoch 1/5 step 300/742 loss=1.8160 elapsed=0.6m
refit epoch 1/5 step 400/742 loss=1.4650 elapsed=0.8m
refit epoch 1/5 step 500/742 loss=1.2286 elapsed=1.0m
refit epoch 1/5 step 600/742 loss=1.0636 elapsed=1.2m
refit epoch 1/5 step 700/742 loss=0.9447 elapsed=1.4m
refit epoch 1/5 step 742/742 loss=0.9010 elapsed=1.5m
refit epoch 2/5 step 100/742 loss=0.1485 elapsed=0.2m
refit epoch 2/5 step 200/742 loss=0.1302 elapsed=0.4m
refit epoch 2/5 step 300/742 loss=0.1268 elapsed=0.6m
refit epoch 2/5 step 400/742 loss=0.1209 elapsed=0.8m
refit epoch 2/5 step 500/742 loss=0.1170 elapsed=1.0m
refit epoch 2/5 step 600/742 loss=0.1155 elapsed=1.2m
refit epoch 2/5 step 700/742 loss=0.1101 elapsed=1.4m
refit epoch 2/5 step 742/742 loss=0.1095 elapsed=1.5m
refit epoch 3/5 step 100/742 loss=0.0654 elapsed=0.2m
refit epoch 3/5 step 200/742 loss=0.0591 elapsed=0.4m
refit epoch 3/5 step 300/742

In [39]:
test_ds=NERWindowDataset(test_text,gold_map=None)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate_windows,num_workers=2,pin_memory=True)
submission_pred=infer_dataset(final_model,test_ds,test_loader,BEST_THRESHOLD)

def validate_prediction(pred,text_map,labels):
    if not isinstance(pred,dict) or not pred: raise ValueError("prediction must be non-empty dict")
    if set(pred)!=set(text_map): raise ValueError(f"ID mismatch missing={len(set(text_map)-set(pred))} extra={len(set(pred)-set(text_map))}")
    allowed=set(labels); n=0
    for did,ents in pred.items():
        if not isinstance(ents,list): raise ValueError(f"{did}: entities must be list")
        seen=set(); text=text_map[did]
        for j,s in enumerate(ents):
            if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f"{did} entity {j}: bad format")
            a,b,l=s[:3]
            if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError("offsets must be ints")
            if not (0<=a<b<=len(text)): raise ValueError(f"{did}: invalid span {a,b}")
            if l not in allowed: raise ValueError(f"{did}: invalid label {l}")
            t=(a,b,l)
            if t in seen: raise ValueError(f"{did}: duplicate {t}")
            seen.add(t); n+=1
    return n

n_spans=validate_prediction(submission_pred,test_text,LABELS)
with open(TEST_PRED_PATH,"w",encoding="utf-8") as f: json.dump(submission_pred,f,ensure_ascii=False)
with zipfile.ZipFile(SUBMISSION_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(TEST_PRED_PATH,arcname="prediction.json")
with zipfile.ZipFile(SUBMISSION_ZIP,"r") as zf:
    names=zf.namelist(); assert names==["prediction.json"]
    obj=json.loads(zf.read("prediction.json").decode("utf-8")); validate_prediction(obj,test_text,LABELS)

print("Predicted test documents:",len(submission_pred))
print("Total predicted spans:",n_spans)
print("ZIP contents:",names)
print("prediction.json:",TEST_PRED_PATH)
print("submission.zip:",SUBMISSION_ZIP)
print("✅ PHASE-2 SUBMISSION PASSED ALL STRUCTURAL CHECKS")


Built 789 windows from 35 docs in 0.0 min
inference batch 50/198
inference batch 100/198
inference batch 150/198
Predicted test documents: 35
Total predicted spans: 3445
ZIP contents: ['prediction.json']
prediction.json: /kaggle/working/phase2_exp1_model4_fullft/prediction.json
submission.zip: /kaggle/working/phase2_exp1_model4_fullft/submission.zip
✅ PHASE-2 SUBMISSION PASSED ALL STRUCTURAL CHECKS


In [40]:
EVALUATOR='#!/usr/bin/env python3\n"""Local evaluator for Phase 2.\n\nUsage:\n  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt\n"""\nimport argparse, json, sys\n\nclass DuplicateJSONKeyError(ValueError): pass\ndef reject_constant(v): raise ValueError(f"Non-standard JSON numeric value \'{v}\' is not allowed.")\ndef pairs_no_duplicates(pairs):\n    out={}; dup=[]\n    for k,v in pairs:\n        if k in out: dup.append(k)\n        out[k]=v\n    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")\n    return out\ndef load_json(path):\n    with open(path,encoding=\'utf-8\') as f:\n        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e\ndef labels(path):\n    with open(path,encoding=\'utf-8\') as f: xs=[x.strip() for x in f if x.strip()]\n    if not xs: raise ValueError(\'labels.txt is empty.\')\n    if len(xs)!=12: raise ValueError(f\'Expected 12 labels, found {len(xs)}.\')\n    if len(set(xs))!=len(xs): raise ValueError(\'labels.txt contains duplicate labels.\')\n    return xs\ndef spans(x, allowed, where):\n    if x is None: return set()\n    if not isinstance(x,list): raise ValueError(f\'{where}: expected a list of entities.\')\n    out=set()\n    for j,s in enumerate(x):\n        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f\'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].\')\n        a,b,l=s[:3]\n        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f\'{where}, entity {j}: start/end must be integers.\')\n        if a<0 or b<=a: raise ValueError(f\'{where}, entity {j}: require 0 <= start < end.\')\n        if not isinstance(l,str) or l not in allowed: raise ValueError(f\'{where}, entity {j}: invalid label {l!r}.\')\n        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f\'{where}, entity {j}: entity_text must be a string.\')\n        t=(a,b,l)\n        if t in out: raise ValueError(f\'{where}, entity {j}: duplicate span {t}.\')\n        out.add(t)\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 2 entity predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); ap.add_argument(\'--labels\',required=True)\n    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)\n    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError(\'Truth and prediction must be non-empty JSON objects.\')\n    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")\n    stats={l:[0,0,0] for l in labs}; allowed=set(labs)\n    for did in truth:\n        g=spans(truth[did],allowed,f\'truth[{did}]\'); p=spans(pred[did],allowed,f\'prediction[{did}]\')\n        for l in labs:\n            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}\n            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)\n    precisions=[]; recalls=[]; f1s=[]\n    print(f\'Examples: {len(truth)}\')\n    for l in labs:\n        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0\n        precisions.append(pr); recalls.append(rc); f1s.append(f)\n        print(f\'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}\')\n    print(f\'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}\')\n    print(f\'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}\')\n    print(f\'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
eval_path=ROOT/"evaluate_phase2.py"
eval_path.write_text(EVALUATOR,encoding="utf-8")
print("Evaluator written:",eval_path)


Evaluator written: /kaggle/working/phase2_exp1_model4_fullft/evaluate_phase2.py


In [41]:
print("KEEP THESE FILES:")
print(" Phase-1 best checkpoint:",MODEL4_BEST_PATH)
print(" Exact tokenizer:",TOKENIZER_PATH)
print(" Phase-2 best heldout:",BEST_NER_PATH)
if FINAL_REFIT_PATH.exists(): print(" Phase-2 final refit:",FINAL_REFIT_PATH)
if TEST_PRED_PATH.exists(): print(" prediction.json:",TEST_PRED_PATH)
if SUBMISSION_ZIP.exists(): print(" submission.zip:",SUBMISSION_ZIP)

print("\nExperiment identity: Model4 + FULL fine-tuning + BILOU token head")
print("This is ONE Phase-2 adaptation experiment; LORMA, KD, and other distinct methods are still required later.")

print(" Tokenizer SHA-256:",tok_sha)
if "BEST_THRESHOLD" in globals():
    print(" Selected confidence threshold(s):",BEST_THRESHOLD)


KEEP THESE FILES:
 Phase-1 best checkpoint: None
 Exact tokenizer: None
 Phase-2 best heldout: /kaggle/working/phase2_exp1_model4_fullft/best_ner.pt
 Phase-2 final refit: /kaggle/working/phase2_exp1_model4_fullft/final_refit_ner.pt
 prediction.json: /kaggle/working/phase2_exp1_model4_fullft/prediction.json
 submission.zip: /kaggle/working/phase2_exp1_model4_fullft/submission.zip

Experiment identity: Model4 + FULL fine-tuning + BILOU token head
This is ONE Phase-2 adaptation experiment; LORMA, KD, and other distinct methods are still required later.
 Tokenizer SHA-256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d
 Selected confidence threshold(s): {'APP': 0.0, 'RESP': 0.0, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.05, 'WIT': 0.0, 'AUTH': 0.1, 'COURT': 0.45, 'STAT': 0.05, 'PREC': 0.05, 'DATE': 0.05, 'CASENO': 0.05}


## Run order & what to send back

1. Run Cells **1–10**, then the tokenizer/checkpoint sanity cell.
   - The **SHA-256 identity check** is the strongest tokenizer check.
   - The causal LM-loss test is an additional heuristic; an extremely high loss is a red flag.
2. Run the data/alignment + **oracle ceiling test**.
   - Send the Fold-1/Fold-2 entity counts, alignment percentage, overlap count, and oracle per-label table.
3. Train the full-fine-tuning experiment.
   - The notebook reports both the best **single global threshold** and the selected **per-label Macro-F1 calibration**.
   - Send the per-epoch history, selected thresholds, and per-label F1.
4. Reload the best held-out checkpoint and inspect Strict Macro-F1.
5. Generate a held-out-selected test submission first with `DO_FINAL_REFIT_ON_ALL=False`.
6. If the held-out pipeline is healthy, optionally refit on Fold 1 + Fold 2 using the already selected epoch count and thresholds, then generate another submission.

### Experiment identity
This remains **Phase 2 Experiment 1 = Model 4 + full fine-tuning**.  
Bidirectional task attention, Viterbi decoding, long-document windowing, and confidence calibration are components of the same full-fine-tuning NER system, not replacements for the later required LORMA / knowledge-distillation / other adaptation experiments.

Useful ablations for analysis (do not count them as distinct adaptation techniques by themselves):
- `BIDIRECTIONAL_ATTENTION=False`
- `USE_VITERBI=False`
- `NER_SEQ_LEN=1024`
